# Population genetic statistics under demography

### Import dependenies

In [ ]:
import msprime
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

# a) Effective population size

In [ ]:
t_max = 2000
dt = 1
times = np.arange(1, t_max + 1)

##### Scenario 1)

In [ ]:
N0 = 1000
r = 0.002

N_exp = N0 * np.exp(r * times)

Ne_exp = []
for t in times:
    inv_mean = np.mean(1 / (N0 * np.exp(r * np.arange(1, t + 1))))
    Ne_exp.append(1 / inv_mean)

Ne_exp = np.array(Ne_exp)

In [ ]:
plt.figure()
plt.plot(times, N_exp, label="N(t)")
plt.plot(times, Ne_exp, label="Ne(t)")
plt.xlabel("Time")
plt.ylabel("Population size")
plt.legend()
plt.show()

##### Scenario 2)

In [ ]:
N_high = 1000
N_low = 100

t_bottleneck_start = 50
t_bottleneck_end = 80

N_bot = []

for t in times:
    if t < t_bottleneck_start:
        N_bot.append(N_high)
    elif t < t_bottleneck_end:
        N_bot.append(N_low)
    else:
        # langsame Erholung (exponentiell zurück)
        recovery_time = t - t_bottleneck_end
        N_recover = N_low * np.exp(0.05 * recovery_time)
        N_bot.append(min(N_recover, N_high))

N_bot = np.array(N_bot)

# Ne für Bottleneck
Ne_bot = []
for t in times:
    inv_mean = np.mean(1 / N_bot[:t])
    Ne_bot.append(1 / inv_mean)

Ne_bot = np.array(Ne_bot)

In [ ]:
plt.figure()
plt.plot(times, N_bot, label="N(t)")
plt.plot(times, Ne_bot, label="Ne(t)")
plt.xlabel("Time")
plt.ylabel("Population size")
plt.ylim(0,N_high*1.1)
plt.legend()
plt.show()

# b) Msprime simulations

### Parameter input

In [ ]:
n = 20
Ne = 10000
growth_rate = 0.0005
mu = 0.0008
num_reps = 1000

pop_config = [
    msprime.PopulationConfiguration(
        sample_size=n,
        initial_size=Ne,
        growth_rate=growth_rate
    )
]

In [ ]:
ts = msprime.simulate(
    population_configurations=pop_config,
    mutation_rate=mu
)
tree=ts.first()
tree.draw_svg(y_axis=True,size=(600,800),node_labels=[])

### Simulation

In [ ]:
sfs_list = []
tajD_list = []
ratio_single_triple = []
ratio_double_triple = []

for _ in tqdm(range(num_reps)):

    ts = msprime.simulate(
        population_configurations=pop_config,
        mutation_rate=mu
    )

    # Site Frequency Spectrum
    sfs = ts.allele_frequency_spectrum(
        polarised=True,
        span_normalise=False
    )
    sfs_list.append(sfs)

    # Tajima's D
    tajD = ts.Tajimas_D()
    tajD_list.append(tajD)

    # Frequenzklassen extrahieren
    if len(sfs) > 3:
        singletons = sfs[1]
        doubletons = sfs[2]
        tripletons = sfs[3]

        if tripletons > 0:
            ratio_single_triple.append(singletons / tripletons)
            ratio_double_triple.append(doubletons / tripletons)

In [ ]:
avg_sfs = np.mean(sfs_list, axis=0)

### Plot

In [ ]:
plt.figure()
plt.bar(range(len(avg_sfs)), avg_sfs)
plt.xlabel("Allelfrequenz")
plt.ylabel("Durchschnittliche Anzahl")
plt.title("Average Site Frequency Spectrum")
plt.show()

In [ ]:
plt.figure()
plt.boxplot([ratio_single_triple, ratio_double_triple],
            labels=["singletons/tripletons", "doubletons/tripletons"])
plt.ylabel("Ratio")
plt.title("SFS-Ratios")
plt.show()

In [ ]:
plt.figure()
plt.boxplot(tajD_list)
plt.ylim(-3,3)
plt.ylabel("Tajima's D")
plt.title("Tajima's D")
plt.show()